# knot — 01: deploy

Build a spec → see the SQL knot generates → execute it → look at the
tables. Subsequent notebooks (02_ingest, 03_query, ...) cover the
runtime concerns.

In [2]:
import uuid

import pandas as pd
import psycopg

In [3]:
# The spec lives in ``movies_spec.py`` next to this notebook — same
# pattern as a real deployment, where the workers + service API all
# import a shared spec module. See that file for the actual class +
# source + binding declarations.
from movies_spec import spec

spec

Spec(identifier_slot_name='canonical_id', classes=[OntologyClass(name='Person', kind=<ClassKind.CONCRETE: 'concrete'>, is_a=None, mixins=[], slots=[Slot(name='canonical_id', type=<Primitive.TEXT: 'text'>, identifier=True, required=True, description=None), Slot(name='name', type=<Primitive.TEXT: 'text'>, identifier=False, required=True, description=None), Slot(name='birth_country', type=<Primitive.TEXT: 'text'>, identifier=False, required=False, description=None)], description=None), OntologyClass(name='Movie', kind=<ClassKind.CONCRETE: 'concrete'>, is_a=None, mixins=[], slots=[Slot(name='canonical_id', type=<Primitive.TEXT: 'text'>, identifier=True, required=True, description=None), Slot(name='title', type=<Primitive.TEXT: 'text'>, identifier=False, required=True, description=None), Slot(name='year', type=<Primitive.INTEGER: 'integer'>, identifier=False, required=False, description=None), Slot(name='director', type=ClassRef(target=OntologyClass(name='Person', kind=<ClassKind.CONCRETE: 

In [4]:
# Host plumbing — psycopg connection + a fresh per-run schema name
# so re-running the notebook never collides with prior runs. We don't
# create the schema here: ``Spec.ddl()`` emits ``CREATE SCHEMA IF NOT
# EXISTS`` as its first statement.
pg = psycopg.connect(
    host="localhost",
    port=5433,
    user="knot",
    password="knot",
    dbname="knot",
    autocommit=True,
)
schema = f"knot_play_{uuid.uuid4().hex[:8]}"
schema

'knot_play_eafdefbe'

In [5]:
# The canonical target schema for the spec, as one SQL script.
# Nothing executes yet — just the text. Notice the order: schema
# → weight table → canonical tables → bindings tables → indexes
# → FK alters → resolved views → all-sources views.
#
# For migrations against a live DB, you wouldn't pg.execute this
# directly — you'd pipe it through sqldef (or Atlas, dbmate, …)
# to get a reconciling diff. See 03_migration for that loop.
print(spec.ddl(schema=schema))

CREATE SCHEMA IF NOT EXISTS knot_play_eafdefbe;

CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS knot_play_eafdefbe.source_weight (
    source_name text NOT NULL,
    class_name  text NOT NULL,
    slot_name   text NOT NULL,
    weight      double precision NOT NULL,
    PRIMARY KEY (source_name, class_name, slot_name)
);

CREATE TABLE IF NOT EXISTS knot_play_eafdefbe.person (
    canonical_id text NOT NULL,
    name text NOT NULL,
    birth_country text,
    PRIMARY KEY (canonical_id)
);

CREATE TABLE IF NOT EXISTS knot_play_eafdefbe.person_bindings (
    source_name text NOT NULL,
    source_identifier text NOT NULL,
    canonical_id text,
    name text,
    birth_country text,
    raw_payload jsonb NOT NULL DEFAULT '{}'::jsonb,
    er_metadata jsonb NOT NULL DEFAULT '{}'::jsonb,
    valid_from timestamptz NOT NULL DEFAULT now(),
    valid_to timestamptz,
    PRIMARY KEY (source_name, source_identifier, valid_from)
);

CREATE INDEX IF NOT EXISTS person_bindings_cur

In [6]:
# First deploy against an empty schema: just run the script. Every
# statement is idempotent (IF NOT EXISTS / CREATE OR REPLACE), so
# re-running is a no-op.
pg.execute(spec.ddl(schema=schema))

<psycopg.Cursor [COMMAND_OK] [IDLE] (host=localhost port=5433 database=knot) at 0x782826fe3d10>

In [7]:
# What landed? Ask postgres via information_schema. For Person +
# Movie we expect: 1 invariant table (source_weight), 2 canonical
# tables, 2 bindings tables, 2 resolved views, 2 all-sources views.
introspect = pd.read_sql_query(
    """
    SELECT t.table_name, t.table_type,
           c.column_name, c.data_type, c.is_nullable
    FROM information_schema.tables t
    JOIN information_schema.columns c USING (table_schema, table_name)
    WHERE t.table_schema = %(schema)s
    ORDER BY t.table_type, t.table_name, c.ordinal_position
    """,
    pg,
    params={"schema": schema},
)
introspect

/tmp/ipykernel_2732789/4088920113.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  introspect = pd.read_sql_query(


,table_name,table_type,column_name,data_type,is_nullable
0,movie,BASE TABLE,canonical_id,text,NO
1,movie,BASE TABLE,title,text,NO
2,movie,BASE TABLE,year,integer,YES
3,movie,BASE TABLE,director,text,YES
4,movie,BASE TABLE,title_embedding,USER-DEFINED,YES
5,movie_bindings,BASE TABLE,source_name,text,NO
6,movie_bindings,BASE TABLE,source_identifier,text,NO
7,movie_bindings,BASE TABLE,canonical_id,text,YES
8,movie_bindings,BASE TABLE,title,text,YES
9,movie_bindings,BASE TABLE,year,integer,YES
